# Token Code Usage Under Dataset Shift

This notebook analyzes how the tokenizer codebook is used on ImageNet-V2 for the two reconstruction models: LlamaGen and VQGAN. The goal is not reconstruction quality here; the goal is vocabulary usage: which discrete codes appear, how concentrated the usage is, and whether usage has spatial structure on the token grid.


## Experiment Setup

Each model encoded the same ImageNet-V2 validation-style dataset. For every image, the encoder produced a `16 x 16` grid of discrete code IDs. The export scripts aggregated the heavy data outside the notebook:

- `global_counts.npy`: total count for each code ID across all images and positions.
- `position_counts.npy`: count for each `(code_id, row, col)` token-grid location.
- `usage.csv`: one row per code ID with count, frequency, activity flag, and rank.
- `summary.json`: compact run-level statistics.

The notebook should stay lightweight: load the precomputed arrays, show the main summaries, and visualize the important differences.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_ROOT = Path('../code_usage_results')
RUNS = {
    'LlamaGen': RESULTS_ROOT / 'llamagen' / 'imagenet_v2',
    'VQGAN': RESULTS_ROOT / 'vqgan' / 'imagenet_v2',
}


In [ ]:
def load_run(run_dir):
    summary = json.loads((run_dir / 'summary.json').read_text())
    global_counts = np.load(run_dir / 'global_counts.npy')
    position_counts = np.load(run_dir / 'position_counts.npy')
    usage = pd.read_csv(run_dir / 'usage.csv')
    return {
        'summary': summary,
        'global_counts': global_counts,
        'position_counts': position_counts,
        'usage': usage,
    }

runs = {name: load_run(path) for name, path in RUNS.items()}


## Quick Validation

A small consistency check is enough here: the global count total should equal the positional count total, and both should equal `n_images * H * W`.


In [ ]:
validation_rows = []
for model, run in runs.items():
    summary = run['summary']
    global_total = int(run['global_counts'].sum())
    position_total = int(run['position_counts'].sum())
    h, w = summary['token_grid_hw']
    expected_total = summary['n_images'] * h * w
    validation_rows.append({
        'model': model,
        'global_total': global_total,
        'position_total': position_total,
        'expected_total': expected_total,
        'ok': global_total == position_total == expected_total,
    })

pd.DataFrame(validation_rows)


## Run-Level Summary

The most direct signal is codebook utilization: how many codes are active, how many are dead, and how concentrated the usage distribution is. Perplexity is the effective number of codes being used after accounting for frequency imbalance.


In [ ]:
summary_rows = []
for model, run in runs.items():
    s = run['summary']
    summary_rows.append({
        'model': model,
        'dataset': s['dataset'],
        'n_images': s['n_images'],
        'active_codes': s['active_codes'],
        'dead_codes': s['dead_codes'],
        'active_fraction_full': s['active_fraction_full'],
        'perplexity': s['perplexity'],
        'top_10_mass': s['top_10_mass'],
        'top_100_mass': s['top_100_mass'],
        'top_500_mass': s['top_500_mass'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


## Active Codebook Size

This plot shows the headline difference: LlamaGen uses the full codebook on ImageNet-V2, while VQGAN uses a much smaller active subset.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary_df['model'], summary_df['active_codes'], color=['C0', 'C1'], alpha=0.85)
ax.axhline(16384, linestyle='--', color='0.35', linewidth=1, label='Full codebook')
ax.set_ylabel('Active codes')
ax.set_title('Active code count on ImageNet-V2')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
plt.show()


## Usage Concentration

Sorted usage curves show whether the model spreads probability mass across many codes or relies on a small subset. A steeper curve means more concentration.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model, run in runs.items():
    counts = run['global_counts']
    freqs = np.sort(counts / counts.sum())[::-1]
    ax.plot(np.arange(1, len(freqs) + 1), freqs, label=model, linewidth=2)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Code rank')
ax.set_ylabel('Frequency')
ax.set_title('Sorted code usage distribution')
ax.grid(alpha=0.25, which='both')
ax.legend(frameon=False)
plt.show()


## Top-Code Mass

This compresses concentration into three numbers: how much of all token usage is explained by the top 10, 100, and 500 codes.


In [ ]:
mass_df = summary_df.melt(
    id_vars='model',
    value_vars=['top_10_mass', 'top_100_mass', 'top_500_mass'],
    var_name='bucket',
    value_name='mass',
)

fig, ax = plt.subplots(figsize=(8, 4))
for offset, model in [(-0.18, 'LlamaGen'), (0.18, 'VQGAN')]:
    sub = mass_df[mass_df['model'] == model]
    x = np.arange(len(sub)) + offset
    ax.bar(x, sub['mass'], width=0.36, label=model, alpha=0.85)

ax.set_xticks(np.arange(3))
ax.set_xticklabels(['Top 10', 'Top 100', 'Top 500'])
ax.set_ylabel('Fraction of all token usage')
ax.set_title('Usage mass captured by top-ranked codes')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False)
plt.show()


## Positional Entropy

For each token-grid location, compute entropy of `P(code | position)`. Higher entropy means that location uses a more diverse set of codes. This gives a first look at whether code usage diversity varies across the spatial grid.


In [ ]:
def position_entropy(position_counts):
    totals = position_counts.sum(axis=0, keepdims=True)
    probs = np.divide(position_counts, totals, out=np.zeros_like(position_counts, dtype=np.float64), where=totals > 0)
    log_probs = np.zeros_like(probs)
    mask = probs > 0
    log_probs[mask] = np.log(probs[mask])
    return -(probs * log_probs).sum(axis=0)

pos_entropy = {model: position_entropy(run['position_counts']) for model, run in runs.items()}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
vmin = min(arr.min() for arr in pos_entropy.values())
vmax = max(arr.max() for arr in pos_entropy.values())

for ax, (model, entropy_grid) in zip(axes, pos_entropy.items()):
    im = ax.imshow(entropy_grid, vmin=vmin, vmax=vmax, cmap='viridis')
    ax.set_title(model)
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes, fraction=0.046, pad=0.04, label='Entropy')
fig.suptitle('Per-position code entropy on ImageNet-V2')
plt.show()


## Initial Reading

The current ImageNet-V2 result is already informative: LlamaGen uses nearly the entire codebook with high effective perplexity, while VQGAN uses a small active subset. The next useful step is to run the same exports on ImageNet and compare ImageNet vs ImageNet-V2 directly within each model.
